# 02 — Cozy Ambient Sounds (Second Teammate) — SoundScape AI

**Google Colab | Conditional VAE | Independent notebook**

Trains a generative deep-learning model for **relaxing ambient sounds**. This is a corrected, self-contained continuation of the original ESC-50 exploration. The model is trained, evaluated, and used to generate 5-second WAV files in **this single notebook**.

- Dataset: [ESC-50](https://github.com/karolpiczak/ESC-50) (40 clips/class, official folds).
- Silent/quiet regions are **audited but not blindly removed**.
- ESC-50 is a *small prototype dataset*; Griffin–Lim reconstruction may sound unnatural. We do **not** claim pleasant audio without listener testing.
- Later, the platform combines layers from **two independent trained models**; volume weights are relative gains, not perceptual percentages.

## 1 — Setup and data

In Google Colab choose **Runtime → Change runtime type → T4 GPU** if available. Run the cells from top to bottom.

In [ ]:
%pip -q install librosa pandas soundfile matplotlib
import subprocess, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa, soundfile as sf
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from IPython.display import Audio, display

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if not Path("ESC-50/meta/esc50.csv").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/karolpiczak/ESC-50.git"], check=True)

## 2 — Classes and split

Official ESC-50 folds **1–3 = training**, **4 = validation**, **5 = untouched test**. We avoid using test examples to select parameters. Both notebooks use the same audio specification so their generated WAV files can later be mixed.

In [ ]:
CLASSES = ['crackling_fire', 'crickets', 'insects', 'chirping_birds', 'water_drops']
CHECKPOINT = "cozy_cvae.pt"
SR = 16000
SECONDS = 5
N_SAMPLES = SR * SECONDS
N_MELS = 64
N_FFT = 1024
HOP = 256
FRAMES = 320  # 64 x 320 is divisible by 16 in both axes
DB_MIN, DB_MAX = -80.0, 20.0
LATENT_DIM = 64
BATCH_SIZE = 16
EPOCHS = 20
LEARNING_RATE = 1e-3
BETA_MAX = 0.001

meta = pd.read_csv("ESC-50/meta/esc50.csv")
meta = meta[meta["category"].isin(CLASSES)].copy()
print(meta.groupby(["category", "fold"]).size().unstack(fill_value=0))
assert len(meta) == 40 * len(CLASSES), "Unexpected ESC-50 class counts"

## 3 — Audio quality audit: duration, silent frames, and zero samples

ESC-50 files normally last five seconds, but some **contain large quiet intervals**. A zero-heavy recording is not necessarily an invalid sample: it may represent distant thunder or sparse insect sounds. Keep these records and **listen before excluding**. To detect genuinely short originals, measure duration *before* padding.

In [ ]:
def load_wave(path):
    y, _ = librosa.load(str(path), sr=SR, mono=True)
    original_duration = len(y) / SR
    y = librosa.util.fix_length(y, size=N_SAMPLES)
    return y.astype(np.float32), original_duration

def inspect(path):
    y, duration = load_wave(path)
    rms_frames = librosa.feature.rms(y=y, frame_length=1024, hop_length=256)[0]
    return {
        "duration_s": duration,
        "rms": float(np.sqrt(np.mean(y**2))),
        "peak": float(np.max(np.abs(y))),
        "quiet_ratio": float(np.mean(rms_frames < 0.001)),
        "zero_ratio": float(np.mean(y == 0)),
    }

records = []
for row in meta.itertuples():
    records.append({"filename": row.filename, "category": row.category,
                    "fold": row.fold, **inspect(Path("ESC-50/audio") / row.filename)})
audit = pd.DataFrame(records)
print("Audited:", len(audit))
print("Short originals:", int((audit.duration_s < 4.99).sum()))
print("Very low global RMS:", int((audit.rms < 0.001).sum()))
display(audit.sort_values("quiet_ratio", ascending=False).head(12))
# No automatic removal. Inspect flagged samples by listening if needed.

## 4 — Fixed mel-spectrogram features

**Important fix:** The earlier starter's 256 mel frames only covered ~4 seconds of a 5-second clip. Here we use **320 frames** so the model represents the full ~5-second duration. Also, a *fixed dB reference* preserves comparable levels between recordings; it does not normalize each sample independently.

In [ ]:
def wave_to_mel(y):
    mel_power = librosa.feature.melspectrogram(
        y=y, sr=SR, n_fft=N_FFT, hop_length=HOP,
        n_mels=N_MELS, power=2.0)
    mel_db = librosa.power_to_db(mel_power, ref=1.0)
    feature = np.clip((mel_db - DB_MIN) / (DB_MAX - DB_MIN), 0, 1)
    feature = librosa.util.fix_length(feature, size=FRAMES, axis=1)
    return feature.astype(np.float32)

def mel_to_wave(feature):
    mel_db = np.asarray(feature, dtype=np.float32) * (DB_MAX - DB_MIN) + DB_MIN
    mel_power = librosa.db_to_power(mel_db, ref=1.0)
    return librosa.feature.inverse.mel_to_audio(
        mel_power, sr=SR, n_fft=N_FFT, hop_length=HOP,
        n_iter=48, power=2.0, length=N_SAMPLES).astype(np.float32)

class ESC50Features(Dataset):
    def __init__(self, folds):
        rows = meta[meta.fold.isin(folds)].reset_index(drop=True)
        xs, ys = [], []
        for row in rows.itertuples():
            y, _ = load_wave(Path("ESC-50/audio") / row.filename)
            xs.append(wave_to_mel(y))
            ys.append(CLASSES.index(row.category))
        self.x = torch.from_numpy(np.stack(xs))[:, None]  # (N,1,64,320)
        self.y = torch.tensor(ys, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.x[i], self.y[i]

train_ds = ESC50Features([1,2,3])
val_ds = ESC50Features([4])
test_ds = ESC50Features([5])
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)
print("Train, validation, test:", len(train_ds), len(val_ds), len(test_ds))
assert tuple(train_ds.x.shape[1:]) == (1,64,320)
plt.figure(figsize=(10,3))
plt.imshow(train_ds.x[0,0], aspect="auto", origin="lower")
plt.title("Example mel features (fixed dB scaling)")
plt.show()

## 5 — Conditional convolutional VAE

The encoder learns **μ** and **log σ²** for a latent Gaussian distribution conditioned on the sound class. The decoder generates a mel representation from sampled **z** and a class. Optimize **mel L1 + β × KL** with KL warm-up.

In [ ]:
class ConditionalVAE(nn.Module):
    def __init__(self, n_classes, latent_dim=64):
        super().__init__()
        self.n_classes = n_classes
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(
            nn.Conv2d(1,16,4,2,1), nn.ReLU(),
            nn.Conv2d(16,32,4,2,1), nn.ReLU(),
            nn.Conv2d(32,64,4,2,1), nn.ReLU(),
            nn.Conv2d(64,64,4,2,1), nn.ReLU())
        self.hidden = 64 * 4 * 20  # (1,64,320) -> (64,4,20)
        self.mu = nn.Linear(self.hidden + n_classes, latent_dim)
        self.logvar = nn.Linear(self.hidden + n_classes, latent_dim)
        self.proj = nn.Linear(latent_dim + n_classes, self.hidden)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64,64,4,2,1), nn.ReLU(),
            nn.ConvTranspose2d(64,32,4,2,1), nn.ReLU(),
            nn.ConvTranspose2d(32,16,4,2,1), nn.ReLU(),
            nn.ConvTranspose2d(16,1,4,2,1), nn.Sigmoid())
    def condition(self, labels):
        return F.one_hot(labels, num_classes=self.n_classes).float()
    def encode(self, x, labels):
        h = torch.cat([self.encoder(x).flatten(1), self.condition(labels)], dim=1)
        return self.mu(h), self.logvar(h)
    def decode(self, z, labels):
        h = torch.cat([z, self.condition(labels)], dim=1)
        h = self.proj(h).reshape(-1,64,4,20)
        return self.decoder(h)
    def forward(self, x, labels):
        mu, logvar = self.encode(x, labels)
        z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
        return self.decode(z, labels), mu, logvar

def vae_loss(pred, target, mu, logvar, beta):
    reconstruction = F.l1_loss(pred, target)
    kl = -0.5 * (1 + logvar - mu.square() - logvar.exp()).sum(dim=1).mean()
    return reconstruction + beta * kl, reconstruction, kl

model = ConditionalVAE(len(CLASSES), LATENT_DIM).to(DEVICE)
x, labels = next(iter(train_loader))
with torch.no_grad():
    pred, mu, logvar = model(x[:2].to(DEVICE), labels[:2].to(DEVICE))
assert pred.shape == x[:2].shape
print("Model shape test passed:", pred.shape)

## 6 — Train and validate (save best weights)

Train on folds 1–3 only. The validation fold selects the best checkpoint; the test fold remains untouched until the next section.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
history = []
best_validation = float("inf")
for epoch in range(1, EPOCHS + 1):
    beta = BETA_MAX * min(epoch / 5, 1.0)
    model.train()
    running = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        pred, mu, logvar = model(xb, yb)
        loss, rec, kl = vae_loss(pred, xb, mu, logvar, beta)
        loss.backward()
        optimizer.step()
        running += loss.item() * len(xb)
    model.eval()
    validation = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            pred, mu, logvar = model(xb, yb)
            loss, _, _ = vae_loss(pred, xb, mu, logvar, beta)
            validation += loss.item() * len(xb)
    train_loss = running / len(train_ds)
    val_loss = validation / len(val_ds)
    history.append([epoch, train_loss, val_loss])
    if val_loss < best_validation:
        best_validation = val_loss
        torch.save({
            "model": model.state_dict(), "classes": CLASSES, "latent_dim": LATENT_DIM,
            "sample_rate": SR, "seconds": SECONDS, "n_fft": N_FFT, "hop": HOP,
            "n_mels": N_MELS, "frames": FRAMES, "db_min": DB_MIN,
            "db_max": DB_MAX, "epoch": epoch, "validation_loss": val_loss,
        }, CHECKPOINT)
    print(f"Epoch {epoch:02d}: train={train_loss:.4f} | val={val_loss:.4f}")

pd.DataFrame(history, columns=["epoch", "train", "validation"]).plot(x="epoch", grid=True)
plt.title("CVAE training / validation loss")
plt.show()
print("Saved best checkpoint:", CHECKPOINT)

## 7 — Evaluate on held-out fold and generate new WAV

**Do not claim production quality:** small data and Griffin–Lim make generation sound rough. The held-out mel L1 value measures reconstruction, **not listener comfort**. Inspect WAV files and compare with real ESC-50 examples.

In [ ]:
weights = torch.load(CHECKPOINT, map_location=DEVICE, weights_only=True)
model.load_state_dict(weights["model"])
model.eval()
reconstruction_sum = 0.0
with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        pred, mu, logvar = model(xb, yb)
        reconstruction_sum += float(F.l1_loss(pred, xb)) * len(xb)
print("Test mel L1 (held out):", reconstruction_sum / len(test_ds))

def generate_sound(class_name, seed=42):
    if class_name not in CLASSES:
        raise ValueError("Choose from: " + str(CLASSES))
    generator = torch.Generator().manual_seed(seed)
    z = torch.randn(1, LATENT_DIM, generator=generator).to(DEVICE)
    label = torch.tensor([CLASSES.index(class_name)], device=DEVICE)
    with torch.no_grad():
        mel = model.decode(z, label)[0,0].cpu().numpy()
    wave = mel_to_wave(mel)
    peak = float(np.max(np.abs(wave)))
    if peak > 1e-7:
        wave = 0.85 * wave / peak  # playback convenience, not training preprocessing
    return wave

for i, name in enumerate(['crackling_fire', 'chirping_birds', 'water_drops']):
    y = generate_sound(name, seed=SEED + i)
    sf.write(f"{name}_generated.wav", y, SR)
    print(name, "RMS:", float(np.sqrt(np.mean(y*y))))
    display(Audio(y, rate=SR))

print("Files available in Colab Files pane:", CHECKPOINT)

## 8 — Download / integration contract

Use the **Colab Files** sidebar to download the `.pt` checkpoint and generated `.wav` examples. Both notebooks export the same waveform format: **mono, 16,000 Hz, 5 seconds**. The planned mixer will load the checkpoints separately and mix generated WAV outputs with independent gain controls (e.g., rain 20, fire 70, birds 10).

For future research: expand to licensed [FSD50K](https://zenodo.org/records/4060432) data, compare with a strong baseline, perform blind listening tests, and improve the waveform reconstruction with a suitable vocoder. Do not interpret the requested mixing ratio as an exact measured perceptual proportion.